In [4]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import os

# 1. Load Pretrained Model và Processor từ Hugging Face
print("Đang tải CLIP model (ViT-B/32)...")
model_id = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id, use_fast=True)
print("Tải model thành công!\n")

# 2. Chuẩn bị dữ liệu đầu vào
image_dir = "./sample_images" 

image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.webp'))][:10]
images = [Image.open(path).convert("RGB") for path in image_paths]
print(f"Đã load thành công {len(images)} ảnh.")


# 10 đoạn text tương ứng để test Semantic Search
texts = [
    "a photo of a cat",
    "a dog playing in the grass",
    "beautiful sunset on the beach",
    "a red car on the street",
    "a cup of coffee on the table",
    "people walking in the city",
    "a laptop on a wooden desk",
    "a delicious pizza with cheese",
    "snow covered mountains",
    "a close up of a beautiful flower"
]

# 3. Xử lý đầu vào và truyền qua Model
if images:
    print("\nBắt đầu trích xuất vector...")
    
    # Processor sẽ tự động resize, normalize ảnh và tokenize text
    inputs = processor(
        text=texts, 
        images=images, 
        return_tensors="pt",
        padding=True         
    )

    # Truyền inputs vào mô hình
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. Quan sát Output
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds

    print("\n--- KẾT QUẢ ĐẦU RA ---")
    print(f"Số lượng ảnh đầu vào: {len(images)}")
    print(f"Số lượng text đầu vào: {len(texts)}")
    
    # Kích thước chuẩn của CLIP ViT-B/32 vector là 512 chiều
    print(f"\nShape của Image Embeddings: {image_embeds.shape}") 
    
    print(f"Shape của Text Embeddings:  {text_embeds.shape}")  

    # Demo tính điểm tương đồng (Similarity Score) giữa text và ảnh
    logits_per_text = outputs.logits_per_text 
    probs_text_to_image = logits_per_text.softmax(dim=1)
    
    print(f"\nShape của Ma trận xác suất (Text-to-Image): {probs_text_to_image.shape}")

Đang tải CLIP model (ViT-B/32)...
Tải model thành công!

Đã load thành công 10 ảnh.

Bắt đầu trích xuất vector...

--- KẾT QUẢ ĐẦU RA ---
Số lượng ảnh đầu vào: 10
Số lượng text đầu vào: 10

Shape của Image Embeddings: torch.Size([10, 512])
Shape của Text Embeddings:  torch.Size([10, 512])

Shape của Ma trận xác suất (Text-to-Image): torch.Size([10, 10])
